# Getting the Chromosome Name from Ensemble Gene Name

## Get the Ensemble Gene Name

In [1]:
import pandas as pd
import polars as pl
import os

In [2]:
DATASET_PATH = '../dataset/E003/'

In [3]:
gene_exp = pl.read_csv(os.path.join(DATASET_PATH, "57epigenomes.RPKM.pc"), separator='\t', truncate_ragged_lines=True)

In [4]:
gene_exp.head()

gene_id,E000,E003,E004,E005,E006,E007,E011,E012,E013,E016,E024,E027,E028,E037,E038,E047,E050,E053,E054,E055,E056,E057,E058,E059,E061,E062,E065,E066,E070,E071,E079,E082,E084,E085,E087,E094,E095,E096,E097,E098,E100,E104,E105,E106,E109,E112,E113,E114,E116,E117,E118,E119,E120,E122,E123,E127,E128
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ENSG00000000003""",23.265,43.985,37.413,29.459,21.864,55.649,52.94,71.629,61.292,44.28,63.184,7.49,8.541,0.576,1.393,1.235,5.544,15.933,27.15,6.433,3.812,6.257,10.151,8.898,14.658,0.298,5.605,73.205,20.954,7.645,35.083,6.265,53.039,64.971,9.594,14.46,3.122,13.463,54.677,13.735,2.435,8.833,4.494,36.012,19.252,11.928,5.637,37.989,0.038,42.639,49.983,11.554,11.847,43.723,0.267,13.758,15.818
"""ENSG00000000005""",0.872,1.642,6.498,0.0,0.157,0.003,0.115,0.087,0.055,1.577,0.726,0.0,0.0,0.0,0.0,0.029,0.0,0.051,0.07,0.0,0.0,0.0,0.0,0.0,0.006,0.0,0.0,0.191,0.0,0.018,0.251,0.0,0.566,0.336,0.03,0.0,0.07,0.0,10.67,0.424,0.032,0.524,0.092,0.205,0.134,0.678,0.121,0.0,0.0,0.0,0.0,0.0,0.018,0.0,0.006,0.0,0.0
"""ENSG00000000419""",55.208,35.259,58.308,48.208,37.477,45.923,44.959,40.438,41.97,51.515,35.129,63.304,47.743,45.394,47.041,38.384,37.351,11.078,12.225,27.126,21.572,28.648,44.444,13.179,22.421,28.648,52.753,52.609,15.701,21.769,26.467,7.879,29.927,30.095,32.469,56.167,32.202,26.051,42.731,19.683,67.684,46.172,33.687,39.226,47.562,61.359,54.866,52.215,79.197,107.098,62.811,42.386,54.869,16.652,73.719,56.578,56.371
"""ENSG00000000457""",3.237,2.596,2.345,8.775,2.723,3.7,3.912,5.011,4.158,3.292,3.16,3.683,2.532,7.409,8.577,7.853,17.602,3.295,4.301,3.706,1.523,1.602,3.933,1.877,4.641,5.433,3.417,4.733,3.349,2.222,5.325,2.977,8.497,9.679,5.593,5.731,2.622,3.907,7.649,5.455,10.873,2.529,2.811,6.044,4.526,8.791,5.484,4.829,11.082,8.814,2.646,2.483,2.527,2.549,7.651,4.967,3.714
"""ENSG00000000460""",7.299,6.649,7.838,7.324,0.83,5.354,5.94,5.704,6.213,7.551,7.705,1.04,1.213,3.459,3.813,3.424,6.91,3.855,4.401,2.896,3.246,3.31,6.491,2.782,2.799,2.292,1.137,0.942,4.716,1.12,1.487,1.611,3.408,3.58,0.806,1.25,0.47,1.134,1.69,1.126,0.518,0.717,0.694,1.893,1.952,3.137,1.631,8.001,13.743,25.369,3.373,4.646,2.179,4.099,22.103,3.29,2.491


In [68]:
ensg_list = pl.Series(gene_exp.select(["gene_id"])).to_list()
print(len(ensg_list))

19795


## Download the Chromosome Name from BioMart

In [75]:
from biomart import BiomartServer
import pandas as pd

In [84]:
def get_ensemble_data(gene_ids, batch_size=100):
    # Connect to the GRCh37 server
    server = BiomartServer("http://grch37.ensembl.org/biomart")
    
    # Select the dataset
    dataset = server.datasets['hsapiens_gene_ensembl']
    
    all_results = []
    
    # Process genes in batches
    for i in range(0, len(gene_ids), batch_size):
        batch = gene_ids[i:i+batch_size]
        print(f"Processing at i = {i}")
        # Prepare and execute the query
        response = dataset.search({
            'attributes': [
                'ensembl_gene_id',
                'external_gene_name',
                'chromosome_name',
                'start_position',
                'end_position',
                'strand'
            ],
            'filters': {
                'ensembl_gene_id': batch
            }
        })

        response_list = list(response.iter_lines())
        print(f"Requested: {batch_size}, Response: {len(response_list)}")

        with open('biomart_response.txt', 'a') as f:
            for line in response_list:
                f.write(f"{line.decode()}\n")

In [85]:
get_ensemble_data(ensg_list)

Processing at i = 0
Requested: 100, Response: 100
Processing at i = 100
Requested: 100, Response: 100
Processing at i = 200
Requested: 100, Response: 100
Processing at i = 300
Requested: 100, Response: 100
Processing at i = 400
Requested: 100, Response: 100
Processing at i = 500
Requested: 100, Response: 100
Processing at i = 600
Requested: 100, Response: 100
Processing at i = 700
Requested: 100, Response: 100
Processing at i = 800
Requested: 100, Response: 100
Processing at i = 900
Requested: 100, Response: 100
Processing at i = 1000
Requested: 100, Response: 100
Processing at i = 1100
Requested: 100, Response: 100
Processing at i = 1200
Requested: 100, Response: 100
Processing at i = 1300
Requested: 100, Response: 100
Processing at i = 1400
Requested: 100, Response: 100
Processing at i = 1500
Requested: 100, Response: 100
Processing at i = 1600
Requested: 100, Response: 100
Processing at i = 1700
Requested: 100, Response: 100
Processing at i = 1800
Requested: 100, Response: 100
Proce

In [91]:
col_names = ['ensembl_gene_id',
                'external_gene_name',
                'chromosome_name',
                'start_position',
                'end_position',
                'strand']

biomart_df = pd.read_csv("biomart_response.txt", sep='\t', names = col_names)

In [92]:
biomart_df.head()

,ensembl_gene_id,external_gene_name,chromosome_name,start_position,end_position,strand
0,ENSG00000000003,TSPAN6,X,99883667,99894988,-1
1,ENSG00000000005,TNMD,X,99839799,99854882,1
2,ENSG00000000419,DPM1,20,49551404,49575092,-1
3,ENSG00000000457,SCYL3,1,169818772,169863408,-1
4,ENSG00000000460,C1orf112,1,169631245,169823221,1


In [94]:
biomart_df.shape

(19645, 6)

In [95]:
len(ensg_list)

19795

In [96]:
biomart_df.shape[0] - len(ensg_list)

-150

In [97]:
ensg_biomart = biomart_df['ensembl_gene_id'].to_list()

In [99]:
ensg_diff = list(set(ensg_list).difference(ensg_biomart))

In [101]:
len(ensg_diff)

150

In [103]:
ensg_diff

['ENSG00000197689',
 'ENSG00000099749',
 'ENSG00000185068',
 'ENSG00000229377',
 'ENSG00000196690',
 'ENSG00000232317',
 'ENSG00000258832',
 'ENSG00000230344',
 'ENSG00000215019',
 'ENSG00000206197',
 'ENSG00000226511',
 'ENSG00000235153',
 'ENSG00000259210',
 'ENSG00000225607',
 'ENSG00000255953',
 'ENSG00000212950',
 'ENSG00000257606',
 'ENSG00000112664',
 'ENSG00000141048',
 'ENSG00000215897',
 'ENSG00000205233',
 'ENSG00000258865',
 'ENSG00000204561',
 'ENSG00000259131',
 'ENSG00000177461',
 'ENSG00000170941',
 'ENSG00000259765',
 'ENSG00000259566',
 'ENSG00000205873',
 'ENSG00000167744',
 'ENSG00000181877',
 'ENSG00000215204',
 'ENSG00000233488',
 'ENSG00000205036',
 'ENSG00000259355',
 'ENSG00000259391',
 'ENSG00000218328',
 'ENSG00000204823',
 'ENSG00000235471',
 'ENSG00000196333',
 'ENSG00000258852',
 'ENSG00000233389',
 'ENSG00000259253',
 'ENSG00000259220',
 'ENSG00000198230',
 'ENSG00000259741',
 'ENSG00000215494',
 'ENSG00000214117',
 'ENSG00000258974',
 'ENSG00000206171',
